# MAPF-GPT S1 benchmark on Colab

Standalone, resumable evaluation of MAPF-GPT-2M on the S1 MovingAI sample. It does not import `mapf_anytime`. Each completed trial is appended immediately to a JSONL file on Google Drive. Re-running the notebook skips completed trials and retries errors.

Expected Drive layout:

```text
MyDrive/mapf-gpt-s1/
└── movingai/
    ├── instances.txt
    ├── instances/
    ├── maps/
    └── scenarios/
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/mapf-gpt-s1")
DATA_ROOT = DRIVE_ROOT / "movingai"
RESULTS_FILE = DRIVE_ROOT / "mapf_gpt_s1_results.jsonl"
SELECTED_FILE = DRIVE_ROOT / "s1_selected_instances.json"
SUMMARY_FILE = DRIVE_ROOT / "mapf_gpt_s1_results.csv"

TIMEOUT_SECONDS = 40.0
MAX_STEPS = 1000
REPETITIONS = 4
BASE_SEED = 0
SAMPLE_SEED = 1729
REQUIRE_A100 = True

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
required = [
    DATA_ROOT / "instances.txt", DATA_ROOT / "instances",
    DATA_ROOT / "maps", DATA_ROOT / "scenarios",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing Drive inputs:\n" + "\n".join(missing))
print(f"Results will be checkpointed to {RESULTS_FILE}")

## Install the pinned official MAPF-GPT implementation

In [ ]:
import subprocess
import sys

MAPF_GPT_REPO = Path("/content/MAPF-GPT")
MAPF_GPT_COMMIT = "d6307447ddc6bc2b7bf5da4f81d4e0707fbc8fa3"
if not MAPF_GPT_REPO.exists():
    subprocess.check_call([
        "git", "clone", "https://github.com/CognitiveAISystems/MAPF-GPT.git",
        str(MAPF_GPT_REPO),
    ])
subprocess.check_call(["git", "-C", str(MAPF_GPT_REPO), "checkout", MAPF_GPT_COMMIT])
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "cppimport==22.8.2", "pybind11==2.13.1", "setuptools<=79.0.1",
    "huggingface-hub", "gymnasium==0.28.1",
    "pydantic==1.10.22", "loguru", "PyYAML", "typing-extensions",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-deps",
    "pogema==1.4.0", "pogema-toolbox==0.1.1",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-deps",
    "-e", str(MAPF_GPT_REPO),
])
if str(MAPF_GPT_REPO) not in sys.path:
    sys.path.insert(0, str(MAPF_GPT_REPO))
from importlib.metadata import version
print("MAPF-GPT installed at", MAPF_GPT_COMMIT)
print("POGEMA", version("pogema"), "| toolbox", version("pogema-toolbox"))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab, select Runtime > Change runtime type > GPU.")
GPU_NAME = torch.cuda.get_device_name(0)
if REQUIRE_A100 and "A100" not in GPU_NAME.upper():
    raise RuntimeError(f"This benchmark requires an A100; Colab allocated {GPU_NAME!r}")
print("PyTorch:", torch.__version__)
print("GPU:", GPU_NAME)

## Reproduce the S1 instance sample

One instance is selected from every one of the 33 map × 6 density-bucket groups using seed 1729. The selected list is persisted to Drive and verified on every restart.

In [ ]:
import json
import random
import re

INSTANCE_NAME = re.compile(
    r"^(?P<map>.+)-(?:random|even)-\d+-density-(?P<bucket>b[1-6])-n\d+\.json$"
)

def load_manifest(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def relative_manifest_paths():
    instance_list = DATA_ROOT / "instances.txt"
    candidates = [
        line.strip() for line in instance_list.read_text(encoding="utf-8").splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    ]
    groups = {}
    for relative in candidates:
        match = INSTANCE_NAME.fullmatch(Path(relative).name)
        if match:
            key = (match.group("map"), match.group("bucket"))
            groups.setdefault(key, []).append(relative)
    maps = sorted({name for name, _ in groups})
    expected = {(name, f"b{bucket}") for name in maps for bucket in range(1, 7)}
    if len(maps) != 33 or set(groups) != expected:
        raise ValueError(
            f"Expected 33 maps x 6 buckets; found {len(maps)} maps and {len(groups)} groups"
        )
    rng = random.Random(SAMPLE_SEED)
    return [rng.choice(sorted(groups[key])) for key in sorted(groups)]

selection_settings = {
    "sample_seed": SAMPLE_SEED,
    "timeout_seconds": TIMEOUT_SECONDS, "max_steps": MAX_STEPS,
    "repetitions": REPETITIONS, "base_seed": BASE_SEED,
    "mapf_gpt_commit": MAPF_GPT_COMMIT, "model": "2M",
}
if SELECTED_FILE.exists():
    existing = json.loads(SELECTED_FILE.read_text(encoding="utf-8"))
    mismatched = {
        key: (existing.get(key), value)
        for key, value in selection_settings.items()
        if existing.get(key) != value
    }
    if mismatched:
        raise RuntimeError(f"{SELECTED_FILE} belongs to a different benchmark configuration")
    selected = existing.get("instances")
    if not isinstance(selected, list) or len(selected) != 198:
        raise RuntimeError(f"Invalid cached selection in {SELECTED_FILE}")
    print(f"Loaded cached S1 selection from {SELECTED_FILE}")
else:
    selected = relative_manifest_paths()
    selection_record = {**selection_settings, "instances": selected}
    SELECTED_FILE.write_text(json.dumps(selection_record, indent=2), encoding="utf-8")
    print(f"Created and cached S1 selection in {SELECTED_FILE}")
print(f"Selected {len(selected)} instances x {REPETITIONS} seeds = {len(selected) * REPETITIONS} trials")

## Load the model and run

Model loading and the first C++ observation-generator compilation are outside the per-instance 40-second budget. Stop the cell whenever needed; running it again resumes from Drive.

In [ ]:
import os
import time
import traceback

import numpy as np
from pogema import GridConfig, pogema_v0
from mapf_gpt.inference import MAPFGPTInference, MAPFGPTInferenceConfig

PASSABLE = frozenset(".GSW")
WEIGHTS = MAPF_GPT_REPO / "weights" / "MAPF-GPT-2M.pt"
WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
model = MAPFGPTInference(MAPFGPTInferenceConfig(
    path_to_weights=str(WEIGHTS), device="cuda",
))

def resolve_from_manifest(manifest_path, value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else (manifest_path.parent / path).resolve()

def read_problem(relative):
    manifest_path = (DATA_ROOT / relative).resolve()
    metadata = load_manifest(manifest_path)
    agents = int(metadata.get("agents", metadata.get("num_agents", 0)))
    if agents <= 0:
        raise ValueError(f"No positive agent count in {manifest_path}")
    map_path = resolve_from_manifest(manifest_path, metadata["map"])
    scenario_path = resolve_from_manifest(manifest_path, metadata["scenario"])

    lines = map_path.read_text(encoding="utf-8").splitlines()
    header, map_start = {}, None
    for index, line in enumerate(lines):
        if line.strip().lower() == "map":
            map_start = index + 1
            break
        key, separator, value = line.partition(" " )
        if separator:
            header[key.lower()] = value.strip()
    if map_start is None:
        raise ValueError(f"No map section in {map_path}")
    width, height = int(header["width"]), int(header["height"])
    rows = lines[map_start:map_start + height]
    if len(rows) != height or any(len(row) != width for row in rows):
        raise ValueError(f"Invalid map dimensions in {map_path}")

    starts, goals = [], []
    with scenario_path.open(encoding="utf-8") as handle:
        if not handle.readline().strip().startswith("version"):
            raise ValueError(f"No version header in {scenario_path}")
        for line in handle:
            fields = line.split()
            if fields:
                starts.append((int(fields[4]), int(fields[5])))
                goals.append((int(fields[6]), int(fields[7])))
                if len(starts) == agents:
                    break
    if len(starts) != agents:
        raise ValueError(f"Scenario supplies {len(starts)}/{agents} agents")
    return {
        "name": str(metadata.get("name", manifest_path.stem)),
        "map": [[0 if tile in PASSABLE else 1 for tile in row] for row in rows],
        "starts": starts, "goals": goals, "agents": agents,
    }

def trimmed_cost(path):
    end = len(path)
    while end > 1 and path[end - 1] == path[end - 2]:
        end -= 1
    return end - 1

def explicit_grid_config(problem, seed):
    # POGEMA's generator-oriented validators cap generated grids at 1024 and
    # agents at 10000. S1 contains larger explicit MovingAI instances, so
    # initialize valid defaults and then install the already-validated map
    # and coordinates. POGEMA itself supports these explicit dimensions.
    config = GridConfig(
        observation_type="MAPF", on_target="nothing",
        max_episode_steps=MAX_STEPS, seed=seed, obs_radius=5,
        collision_system="soft",
    )
    config.map = problem["map"]
    config.size = max(len(problem["map"]), len(problem["map"][0]))
    config.density = sum(map(sum, problem["map"])) / sum(
        len(row) for row in problem["map"]
    )
    config.agents_xy = [[y, x] for x, y in problem["starts"]]
    config.targets_xy = [[y, x] for x, y in problem["goals"]]
    config.num_agents = problem["agents"]
    return config

def solve_one(problem, seed):
    environment = pogema_v0(grid_config=explicit_grid_config(problem, seed))
    observations, _ = environment.reset()
    model.reset_states()
    model.torch_generator.manual_seed(seed)
    paths = [[tuple(position)] for position in problem["starts"]]
    started = time.perf_counter()
    deadline = started + TIMEOUT_SECONDS
    solved, steps, status = False, 0, "step_limit"
    try:
        for step in range(MAX_STEPS):
            if time.perf_counter() >= deadline:
                status = "timeout"
                break
            actions = model.act(observations)
            observations, _, terminated, truncated, _ = environment.step(actions)
            for path, (row, column) in zip(
                paths, environment.get_agents_xy(ignore_borders=True)
            ):
                path.append((int(column), int(row)))
            steps = step + 1
            if all(terminated):
                solved, status = True, "solved"
                break
            if all(truncated):
                status = "step_limit"
                break
    finally:
        environment.close()
    elapsed = time.perf_counter() - started
    costs = [trimmed_cost(path) for path in paths] if solved else []
    return {
        "status": status, "solved": solved, "wall_seconds": elapsed,
        "steps": steps, "soc": sum(costs) if solved else None,
        "makespan": max(costs, default=None) if solved else None,
    }

def read_results():
    latest = {}
    if RESULTS_FILE.exists():
        for line in RESULTS_FILE.read_text(encoding="utf-8").splitlines():
            try:
                record = json.loads(line)
                latest[int(record["task"])] = record
            except (json.JSONDecodeError, KeyError, ValueError):
                pass
    return latest

def append_result(record):
    with RESULTS_FILE.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, separators=(",", ":")) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

In [ ]:
results = read_results()
total = len(selected) * REPETITIONS
for task in range(total):
    repetition, instance_index = divmod(task, len(selected))
    seed = BASE_SEED + repetition
    previous = results.get(task)
    if previous and previous.get("status") in {"solved", "timeout", "step_limit"}:
        continue
    relative = selected[instance_index]
    problem = None
    try:
        problem = read_problem(relative)
        outcome = solve_one(problem, seed)
        record = {
            "task": task, "instance": problem["name"],
            "manifest": relative, "agents": problem["agents"],
            "repetition": repetition + 1, "seed": seed,
            "gpu": GPU_NAME, **outcome, "error": None,
        }
    except Exception as error:
        record = {
            "task": task,
            "instance": problem["name"] if problem else Path(relative).stem,
            "manifest": relative,
            "agents": problem["agents"] if problem else None,
            "repetition": repetition + 1, "seed": seed,
            "gpu": GPU_NAME, "status": "error", "solved": False,
            "wall_seconds": None, "steps": None, "soc": None,
            "makespan": None,
            "error": f"{type(error).__name__}: {error}",
        }
        traceback.print_exc()
    append_result(record)
    results[task] = record
    solved_so_far = sum(item.get("solved", False) for item in results.values())
    completed = sum(
        item.get("status") in {"solved", "timeout", "step_limit"}
        for item in results.values()
    )
    label = "SOLVED" if record["solved"] else record["status"].upper()
    print(
        f"[{task + 1}/{total}] seed={seed} {record['instance']}: {label} "
        f"({record.get('wall_seconds') or 0:.2f}s); "
        f"overall {solved_so_far}/{completed} solved", flush=True,
    )

print("Run cell finished. Execute the summary cell below.")

## Current summary

In [ ]:
import pandas as pd

latest = read_results()
frame = pd.DataFrame(latest.values()).sort_values("task") if latest else pd.DataFrame()
if frame.empty:
    print("No results yet.")
else:
    frame.to_csv(SUMMARY_FILE, index=False)
    finished = frame[frame.status.isin(["solved", "timeout", "step_limit"])]
    solved = int(finished.solved.sum())
    print(f"Completed: {len(finished)}/{len(selected) * REPETITIONS}")
    print(f"Solved: {solved}/{len(finished)} ({100 * solved / len(finished):.2f}%)" if len(finished) else "Solved: n/a")
    print(f"Errors awaiting retry: {(frame.status == 'error').sum()}")
    print(f"CSV: {SUMMARY_FILE}")
    display(frame.status.value_counts().rename_axis("status").to_frame("trials"))